In [51]:
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader, Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
import os
from langchain_groq import ChatGroq
from dotenv import load_dotenv

import re
import json

from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field, EmailStr
from typing import List, Annotated, Optional

In [22]:
load_dotenv()

True

In [23]:
dir_address = r"C:\Users\sinha\Documents\Project\Resume Builder\Backend\pdf"

# Load PDFs
pdf_loader = DirectoryLoader(
    path=dir_address,
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader
)

# Load Word files
docx_loader = DirectoryLoader(
    path=dir_address,
    glob="**/*.docx",
    loader_cls=Docx2txtLoader
)

In [24]:
# Load documents
pdf_docs = pdf_loader.load()
docx_docs = docx_loader.load()

# Combine both
documents = pdf_docs + docx_docs

In [25]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=8000,
    chunk_overlap=1200
)

In [26]:
docs = text_splitter.split_documents(documents)

## Embedding & Vector Database

In [27]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [28]:
vectordb = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory="vector_db"
)

In [29]:
retriever = vectordb.as_retriever(
    search_type="mmr",  # Max Marginal Relevance - balances relevance and diversity
    search_kwargs={
        "k": 5,  # Return top 5 results for resume context
        "fetch_k": 20,  # Fetch 20 before selecting 5 (for MMR)
        "lambda_mult": 0.25  # Balance between relevance (1) and diversity (0)
    }
)

In [30]:
llm = ChatGroq(
    model = 'llama-3.1-8b-instant',
    api_key= os.getenv("GROQ_API_KEY2")
)

In [31]:
# Create the RAG Q&A chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",  # Simple approach: stuff all retrieved docs into context
    retriever=retriever,
    return_source_documents=True,  # Show which documents were used
    verbose=True  # See the retrieval process
)

# Now you can ask questions about the resumes!
# The system will automatically:
# 1. Find relevant resume chunks using the retriever
# 2. Feed them to the LLM
# 3. LLM extracts and synthesizes the important information

# Example usage:
question = "What are the candidate's key skills and experience?"
result = qa_chain.invoke({"query": question})

# print("Question:", question)
# print("\nAnswer:", result["result"])
# print("\nSources used:")
# for doc in result["source_documents"]:
#     print(f"- {doc.metadata.get('source', 'Unknown')} (page {doc.metadata.get('page', 'N/A')})")



> Entering new RetrievalQA chain...

> Finished chain.


In [32]:
# Helper function for easy Q&A
def ask_resume(question):
    """Ask questions about the resumes in the dataset"""
    result = qa_chain.invoke({"query": question})
    return {
        "question": question,
        "answer": result["result"],
        "sources": [doc.metadata.get('source', 'Unknown') for doc in result["source_documents"]]
    }

In [33]:
response = ask_resume("Mention the skills candinate mentioned and used on the projects")
# print("Q:", response["question"])
# print("A:", response["answer"])
# print("Sources:", response["sources"])



> Entering new RetrievalQA chain...

> Finished chain.


In [37]:
import json

# Load the existing resume schema JSON file
schema_file_path = r"C:\Users\sinha\Documents\Project\Resume Builder\Backend\output\resume_schema.json"

with open(schema_file_path, 'r', encoding='utf-8') as f:
    resume_schema = json.load(f)

## Detailed Prompt Templates for Resume Information Extraction

In [46]:
personal_info_prompt = PromptTemplate(
    template="""
You are an intelligent resume parser.

Extract ONLY the candidate personal contact information from the resume context.

Rules:
- Return ONLY valid JSON.
- Do NOT add explanations or markdown.
- If GitHub is not present return an empty string.
- Follow this exact JSON format.

{{
 "personal_info": {{
  "name": "",
  "email": "",
  "phone": "",
  "linkedin": "",
  "github": ""
 }}
}}

Resume Context:
{context}
""",
    input_variables=["context"]
)

In [47]:
class PersonalInfo(BaseModel):
    name: str
    email: EmailStr
    phone: str
    linkedin: str
    github: Optional[str] = ""


class ResumePersonalSchema(BaseModel):
    personal_info: PersonalInfo

In [48]:
retrieved_docs = retriever.invoke("candidate personal information name email phone linkedin github")

resume_context = " ".join([doc.page_content for doc in retrieved_docs])

In [49]:
response = llm.invoke(
    personal_info_prompt.format(context=resume_context)
)

raw_output = response.content.strip()

In [ ]:
clean_json = re.sub(r"```json|```", "", raw_output).strip()

data = json.loads(clean_json)

In [52]:
validated_data = ResumePersonalSchema(**data)

In [53]:
resume_schema["personal_info"] = validated_data.personal_info.model_dump()

with open(schema_file_path, "w", encoding="utf-8") as f:
    json.dump(resume_schema, f, indent=2, ensure_ascii=False)

- career Objective 

In [34]:
career_objective_prompt = PromptTemplate(
    template="""
You are a professional resume writer.

Write ONLY a career objective paragraph based on the candidate's resume information.

Instructions:
- Write in first person.
- Style similar to: a reflective technical student describing interests, skills, and mindset.
- Focus on field of study, AI/Data Science interests, technical tools, and problem-solving mindset.
- If leadership, mentoring, chess, workshops, or hackathons appear in the context, connect them with analytical thinking or collaboration.
- Tone: professional, thoughtful, and natural.
- Length: 4–6 sentences.
- Output MUST be a single paragraph.
- Do NOT include titles, labels, bullet points, or explanations.
- Return ONLY the paragraph text.

Resume Context:
{context}
""",
    input_variables=["context"]
)

In [39]:
# Generate Career Objective using the prompt template and retriever
# Get relevant resume chunks for career objective generation
retrieved_docs = retriever.invoke("candidate background experience skills education career goals")

# Combine the retrieved text
resume_context = " ".join([doc.page_content for doc in retrieved_docs])

# Generate career objective using the prompt template
career_objective_response = llm.invoke(career_objective_prompt.format(context=resume_context))

# Extract the career objective text
generated_career_objective = career_objective_response.content.strip()

# Update the career_objective field with the generated content
resume_schema['career_objective'] = generated_career_objective.strip()  # Remove any extra whitespace

# Save the updated schema back to the file
with open(schema_file_path, 'w', encoding='utf-8') as f:
    json.dump(resume_schema, f, indent=2, ensure_ascii=False)

- Education 

In [60]:
education_prompt = PromptTemplate(
    template="""
You are an intelligent resume parser.

…

Return the result in this format:

{{
 "education": {{
  "college": {{
   "degree": "",
   "year": "",
   "institute_name": "",
   "board_or_university": "",
   "score_or_cgpa": ""
  }},

  "class_12": {{
   "degree": "",
   "year": "",
   "institute_name": "",
   "board_or_university": "",
   "score_or_cgpa": ""
  }},

  "class_10": {{
   "degree": "",
   "year": "",
   "institute_name": "",
   "board_or_university": "",
   "score_or_cgpa": ""
  }}
 }}

Resume Context:
{context}
""",
    input_variables=["context"]
)

In [61]:
class EducationItem(BaseModel):
    degree: str
    year: str
    institute_name: str
    board_or_university: str
    score_or_cgpa: str


class EducationSchema(BaseModel):
    education: List[EducationItem]

In [62]:
retrieved_docs = retriever.invoke(
    "education degree institute board university cgpa percentage"
)

education_context = " ".join([doc.page_content for doc in retrieved_docs])

In [63]:
response = llm.invoke(
    education_prompt.format(context=education_context)
)

raw_output = response.content.strip()

In [65]:
clean_json = re.sub(r"```json|```", "", raw_output).strip()

# parse the cleaned output safely
try:
    education_data = json.loads(clean_json)
except json.JSONDecodeError as e:
    print("⚠️ JSON parsing failed for education output:", e)
    education_data = {}

# optional: validate with Pydantic schema (if defined)
try:
    validated_education = EducationSchema(**education_data)
    education_data = validated_education.model_dump()
except Exception as e:
    print("⚠️ Education schema validation error:", e)


⚠️ JSON parsing failed for education output: Expecting value: line 1 column 1 (char 0)
⚠️ Education schema validation error: 1 validation error for EducationSchema
education
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing


- Skills Extraction

In [ ]:
# Skills Extraction
skills_prompt = PromptTemplate(
    template="""
You are a professional resume parser. Extract and categorize skills from the resume text.

Guidelines:
- Programming Languages: Python, R, C/C++, Java, HTML/CSS, JavaScript, SQL, etc.
- Databases: Categorize into RDBMS (MySQL, PostgreSQL), Non-RDBMS (MongoDB), Vector Databases (Chroma, Pinecone)
- Libraries/Frameworks: Pandas, NumPy, Matplotlib, Scikit-Learn, PyTorch, TensorFlow, Flask, FastAPI, LangChain, etc.
- Tools: Excel, Power BI, Git/GitHub, Docker, Bash, Shell, etc.
- IDEs: Jupyter notebook, VS Code, PyCharm, Cursor, etc.
- Be comprehensive and accurate

Resume Context:
{context}

Return ONLY a JSON object with categorized skills in this exact format:
{{
 "programming_languages": ["Python", "R", "C/C++"],
 "databases": {{
  "rdbms": ["MySQL", "PostgreSQL"],
  "non_rdbms": ["MongoDB"],
  "vector_databases": ["Chroma", "Pinecone"]
 }},
 "libraries_frameworks": ["Pandas", "NumPy", "Scikit-Learn"],
 "tools": ["Excel", "Power BI", "Git/GitHub", "Docker"],
 "ides": ["Jupyter notebook", "VS Code", "PyCharm"]
}}
""",
    input_variables=["context"]
)

In [ ]:
# Projects and Internships Extraction
projects_internships_prompt = PromptTemplate(
    template="""
You are a professional resume parser. Extract project and internship information from the resume text.

Guidelines:
- For Projects: Extract name, languages used, and description
- For Internship Project Details: Extract role, organization, project link, and description points
- For Internship Programs: Extract organization and program name
- Be detailed and accurate with descriptions
- Preserve technical details and achievements

Resume Context:
{context}

Return ONLY a JSON object with projects and internships in this exact format:
{{
 "projects": [
  {{
   "project_name": "CampusGPT",
   "languages_used": ["HTML", "CSS", "JavaScript", "PHP"],
   "description": "Developed a responsive travel website..."
  }}
 ],
 "internship_project_details": [
  {{
   "role": "AI/Tech Project Manager Intern",
   "organization": "BeFit Naturally",
   "project_link": "https://github.com/Chitranshu0/Automate-post/",
   "description_points": [
    "Developed a FastAPI dashboard...",
    "Built pipelines integrating 10+ data sources...",
    "Engineered API-driven workflows..."
   ]
  }}
 ],
 "internship_programs": [
  {{
   "organization": "Remark Skill Education",
   "program": "Machine Learning using Python"
  }}
 ]
}}
""",
    input_variables=["context"]
)

In [ ]:
# Positions of Responsibility Extraction
positions_prompt = PromptTemplate(
    template="""
You are a professional resume parser. Extract positions of responsibility from the resume text.

Guidelines:
- Extract role, organization, and specific responsibilities
- Include leadership roles, club positions, mentoring activities
- List responsibilities as bullet points
- Be comprehensive with achievements and contributions

Resume Context:
{context}

Return ONLY a JSON array of position objects in this exact format:
[{{
 "role": "Data Science Lead in Google Developer Group on Campus",
 "organization": "Google Developer Group",
 "responsibilities": [
  "Conducted hackathons: Tech-fest (January-2026), Marvels of Machine Learning (December-2025)",
  "Technically support other GDG events...",
  "Co-Lead recruitment for the Club..."
 ]
}}]
""",
    input_variables=["context"]
)

In [ ]:
# Personal Details and Other Information
personal_other_prompt = PromptTemplate(
    template="""
You are a professional resume parser. Extract personal details and other information from the resume text.

Guidelines:
- Extract personal information, achievements, certifications, soft skills
- Be accurate with dates, addresses, and contact information
- Include languages known, interests, hobbies
- Extract training/certifications, achievements, extracurricular activities

Resume Context:
{context}

Return ONLY a JSON object with personal and other information in this exact format:
{{
 "personal_details": {{
  "date_of_birth": "30/06/2004",
  "gender": "Male",
  "father_name": "Mr. Anjay Kumar Sinha",
  "present_address": "CRPF GC, Nayapalli, Bhubaneswar, Odisha, Po: 751012",
  "permanent_address": "Sindri, Dhanbad, Jharkhand, Po: 828128",
  "alternate_phone": "7017003218",
  "languages_known": ["English", "Hindi", "Spanish(A2)"],
  "interests_hobbies": ["Chess", "Reading", "Travelling", "Exploring", "connecting with peoples", "Technology", "Duolingo", "playing Table tennis"]
 }},
 "training_certifications": [
  "OCI AI Foundation & Data Science (Oracle)",
  "R programming & Programming in Modern C++ (NPTEL)",
  "Introduction to Data Science (Cisco)"
 ],
 "achievements_extracurricular": [
  "Secured mainly 1st and 2nd positions on University Chess tournaments",
  "90+ Questions on Leetcode",
  "Secured 1st position on Eureka hosted by PDCS club"
 ],
 "soft_skills": [
  "Problem Solving",
  "Communication",
  "Leadership",
  "Time Management",
  "Teamwork"
 ],
 "signature_section": {{
  "place": "Gunupur, Odisha",
  "date": "",
  "signature": ""
 }}
}}
""",
    input_variables=["context"]
)

In [ ]:
# Master Resume Extraction Template
master_resume_extraction_prompt = PromptTemplate(
    template="""
You are an expert AI resume parser and professional writer. Your task is to extract and structure ALL information from the resume text into a comprehensive JSON format.

Guidelines:
- Extract information with high accuracy and completeness
- Maintain professional formatting and terminology
- Include all technical details, achievements, and experiences
- Structure information logically according to the provided schema
- For arrays, include all relevant items
- For descriptions, preserve technical accuracy and achievements
- If information is not available, use empty strings or empty arrays as appropriate
- Use "personal_info" for contact and basic information
- Format education with separate fields: degree, year, institute, board, score

Resume Context:
{context}

Return ONLY a valid JSON object that matches this exact schema structure:
{schema}

Ensure the JSON is properly formatted and contains all extracted information from the resume.
""",
    input_variables=["context", "schema"]
)

In [ ]:
# Retrieve resume text for career objective generation
# Get relevant chunks about the candidate's background and experience
retrieved_docs = retriever.invoke("candidate background experience skills education")

# Combine the retrieved text
retrieved_resume_text = " ".join([doc.page_content for doc in retrieved_docs])

In [ ]:
# Generate Career Objective using the prompt template and retriever
# Get relevant resume chunks for career objective generation
retrieved_docs = retriever.invoke("candidate background experience skills education career goals")

# Combine the retrieved text
resume_context = " ".join([doc.page_content for doc in retrieved_docs])

# Generate career objective using the prompt template
career_objective_response = llm.invoke(career_objective_prompt.format(context=resume_context))

# Extract the career objective text
generated_career_objective = career_objective_response.content.strip()

In [ ]:
generated_career_objective

'As a detail-driven and analytical professional with a strong passion for Artificial Intelligence and Data Science, I aim to leverage my skills in machine learning, data visualization, and problem-solving to drive business growth and innovation in a collaborative environment. With expertise in programming languages such as Python, R, and C++, I am well-equipped to develop and implement data-driven solutions that address complex problems. Through my experience in leading teams and mentoring peers, I have honed my leadership skills and excel in fostering a culture of collaboration, innovation, and knowledge-sharing. I am excited to contribute my technical expertise and analytical mind to a dynamic organization where I can continue to learn, grow, and make a meaningful impact.'

In [ ]:
2s = ''' 
'As a detail-driven and analytical professional with a strong passion for Artificial Intelligence and Data Science, I aim to leverage my skills in machine learning, data visualization, and problem-solving to drive business growth and innovation in a collaborative environment. With expertise in programming languages such as Python, R, and C++, I am well-equipped to develop and implement data-driven solutions that address complex problems. Through my experience in leading teams and mentoring peers, I have honed my leadership skills and excel in fostering a culture of collaboration, innovation, and knowledge-sharing. I am excited to contribute my technical expertise and analytical mind to a dynamic organization where I can continue to learn, grow, and make a meaningful impact.'
'''